In [ ]:
#| default_exp datasources.mosaico
%load_ext autoreload
%autoreload 2

import sys,os
from pathlib import Path

In [ ]:
# Insert in Path Project Directory
sys.path.insert(0, str(Path().cwd().parent))
os.chdir(Path.cwd().parent / 'extracao')

# Mosaico
> Módulo para encapsular a extração e processamento comum às bases do Mosaico

O cache bruto não armazena a coluna `Log`. Ao carregá-lo, a extração recria essa coluna com `[]`, iniciando um novo histórico de processamento, assim como na extração do MongoDB.

## Agrupamento e preservação de linhas

`Mosaico.aggregate` agrupa pelos campos selecionados, colapsa duplicatas e registra a `Multiplicidade`. `Mosaico.ungrouped` normaliza e ordena sem colapsar duplicatas nem descartar linhas com valores nulos nos campos de agrupamento. A multiplicidade constante igual a 1 é usada apenas durante o processamento e não aparece na saída completa.

In [ ]:
#| export
import os

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from fastcore.foundation import GetAttr

from extracao.datasources.base import Base
from extracao.datasources.connectors import MongoDB

In [ ]:
#| export
load_dotenv(find_dotenv())

True

In [ ]:
#| hide: true
#| eval:false
__file__ = Path.cwd().parent / 'extracao' / 'datasources.py'

In [ ]:
#| export
MONGO_URI: str = os.environ.get("MONGO_URI")


In [ ]:
# | export
class Mosaico(Base, GetAttr):
    """Base Class with the common API from the MOSAICO MongoDB Source"""

    def __init__(self, mongo_uri: str = MONGO_URI, read_cache: bool = False):
        self.read_cache = read_cache
        self.database = "sms"
        self.default = MongoDB(mongo_uri)

    @property
    def collection(self):
        raise NotImplementedError(
            "Subclasses should implement the property 'collection'"
        )

    @property
    def query(self):
        raise NotImplementedError("Subclasses should implement the property 'query'")

    @property
    def projection(self):
        raise NotImplementedError(
            "Subclasses should implement the property 'projection'"
        )

    def _extract(self, collection: str, pipeline: list):
        if self.read_cache:
            df = self._read(f"{self.stem}_raw")
            df["Log"] = "[]"
            return df
        client = self.connect()
        database = client[self.database]
        db_collection = database[collection]
        df = pd.DataFrame(list(db_collection.aggregate(pipeline)), dtype="string")
        # Substitui strings vazias, espaços e listas vazias por nulo
        df = df.replace(r"^\s*$|^\[\]$", pd.NA, regex=True)
        # Create the Log Column
        df["Log"] = "[]"
        return df

    @staticmethod
    def split_designacao(
        df: pd.DataFrame,  # DataFrame com coluna original DesignacaoEmissao
    ) -> (
        pd.DataFrame
    ):  # DataFrame com novas colunas Largura_Emissão(kHz) e Classe_Emissão
        """Parse a bandwidth string
        It returns the numerical component and a character class
        """
        # split then explode
        df["Temp"] = (
            df["Designação_Emissão"]
            .str.strip()
            .str.replace(",", " ")
            .str.strip()
            .str.upper()
            .str.split()
        )

        df = df.explode("Temp", ignore_index=True)
        # Removes empty rows
        df = df[df["Temp"] != "/"]
        df["Temp"] = df["Temp"].astype("string").fillna("")
        # Apply the parse_bw function
        parsed_data = zip(*df["Temp"].apply(Base.parse_bw))
        df["Largura_Emissão(kHz)"], df["Classe_Emissão"] = parsed_data
        df["Largura_Emissão(kHz)"] = df["Largura_Emissão(kHz)"].astype("string")
        df["Classe_Emissão"] = df["Classe_Emissão"].astype("string")
        return df.drop(["Designação_Emissão", "Temp"], axis=1)

    @staticmethod
    def aggregate(
        df: pd.DataFrame,  # DataFrame com os dados de Estações
        agg_cols: list,  # Lista de colunas a serem agrupadas
        log: str,  # Mensagem de log a ser registrada
    ) -> pd.DataFrame:  # DataFrame com os dados duplicados excluídos
        """Exclude and log the duplicated rows
        Columns considered are defined by agg_cols
        """
        df = Mosaico._prepare_rows(df)
        duplicated = df.duplicated(subset=agg_cols, keep="first")
        # I didn't find a better way to do this, the LLMs suggestions were wrong!
        df_sub = df[~duplicated].dropna(subset=agg_cols).reset_index(drop=True)
        grouped = df.groupby(agg_cols, dropna=True, sort=False, observed=True)
        df_sub["Multiplicidade"] = grouped.size().values
        df_sub["#Estação"] = grouped["Estação"].apply(lambda x: list(x)).values
        df_sub["#Estação"] = df_sub["#Estação"].astype("string")
        row_filter = df_sub["Multiplicidade"] > 1
        processing = f"{log}: {agg_cols}"
        return Mosaico.register_log(
            df=df_sub, processing=processing, column="#Estação", row_filter=row_filter
        )

    @staticmethod
    def _prepare_rows(df: pd.DataFrame) -> pd.DataFrame:
        """Normalize and sort rows before aggregation."""
        df["Estação"] = (
            df["Estação"].astype("string").fillna("-1").astype("int", errors="ignore")
        )
        df = df.sort_values("Estação", ignore_index=True)
        df["Largura_Emissão(kHz)"] = pd.to_numeric(
            df["Largura_Emissão(kHz)"], errors="coerce"
        )
        return df

    @staticmethod
    def ungrouped(df: pd.DataFrame) -> pd.DataFrame:
        """Normalize and sort rows without collapsing duplicates."""
        df = Mosaico._prepare_rows(df)
        df["Multiplicidade"] = 1
        return df.reset_index(drop=True)

In [ ]:
#| hide
from nbdev.doclinks import nbdev_export 
nbdev_export()